In [3]:
"""
FOOD PRICE FORECASTING
DATA AUDIT — EXISTING LOCAL DATASET
====================================

Purpose:
    Audit the existing AGMARKNET data already downloaded locally.

Important:
    - This cell does NOT call AGMARKNET.
    - This cell does NOT modify the raw dataset.
    - This cell does NOT attempt the 374 missing crop-month requests.
    - It only reads the existing raw/processed files and reports quality,
      coverage, gaps, duplicates, and consistency.

Run this before moving to Notebook 02.
"""

import csv
from pathlib import Path

import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)


# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


RAW_PRICE_FILE = RAW_DIR / "agmarknet_price_observations.csv"
MONTHLY_PRICE_FILE = PROCESSED_DIR / "monthly_crop_prices.csv"
LOG_FILE = OUTPUT_DIR / "agmarknet_ingestion_log.csv"

AUDIT_OUTPUT_FILE = OUTPUT_DIR / "data_audit_report.txt"
COVERAGE_OUTPUT_FILE = OUTPUT_DIR / "data_audit_crop_month_coverage.csv"


print("=" * 90)
print("FOOD PRICE FORECASTING — EXISTING DATA AUDIT")
print("=" * 90)
print()
print("Project root:", PROJECT_ROOT)
print("Raw file:", RAW_PRICE_FILE)
print("Monthly file:", MONTHLY_PRICE_FILE)
print("Ingestion log:", LOG_FILE)


# ============================================================
# 2. REQUIRED RAW SCHEMA
# ============================================================

RAW_COLUMNS = [
    "Date",
    "Crop",
    "Commodity ID",
    "Commodity Group ID",
    "State",
    "State ID",
    "Market",
    "Market ID",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit",
]

KEY_COLUMNS = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade",
]


# ============================================================
# 3. FILE EXISTENCE CHECK
# ============================================================

print()
print("-" * 90)
print("1. FILE EXISTENCE")
print("-" * 90)

for label, path in [
    ("Raw AGMARKNET file", RAW_PRICE_FILE),
    ("Monthly processed file", MONTHLY_PRICE_FILE),
    ("Ingestion log", LOG_FILE),
]:
    exists = path.exists()

    if exists:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{label}: EXISTS | {size_mb:.2f} MB")
    else:
        print(f"{label}: MISSING")


if not RAW_PRICE_FILE.exists():
    raise FileNotFoundError(
        f"Required raw file was not found:\n{RAW_PRICE_FILE}"
    )


# ============================================================
# 4. RAW CSV STRUCTURE AUDIT
# ============================================================

print()
print("-" * 90)
print("2. RAW CSV STRUCTURE")
print("-" * 90)

total_csv_rows = 0
malformed_rows = 0
field_counts = {}
first_malformed_rows = []

with open(
    RAW_PRICE_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as source:

    reader = csv.reader(source)

    try:
        header = next(reader)
    except StopIteration:
        header = []

    for line_number, row in enumerate(reader, start=2):
        total_csv_rows += 1

        count = len(row)
        field_counts[count] = field_counts.get(count, 0) + 1

        if count != len(RAW_COLUMNS):
            malformed_rows += 1

            if len(first_malformed_rows) < 10:
                first_malformed_rows.append(
                    {
                        "line": line_number,
                        "field_count": count,
                        "row_preview": row[:18],
                    }
                )


print("Header:")
print(header)

print()
print("Expected fields:", len(RAW_COLUMNS))
print("Actual header fields:", len(header))
print("Total CSV data rows:", f"{total_csv_rows:,}")
print("Malformed rows:", f"{malformed_rows:,}")

print()
print("Field-count distribution:")
for count, frequency in sorted(field_counts.items()):
    print(f"  {count} fields: {frequency:,} rows")

if first_malformed_rows:
    print()
    print("First malformed rows:")
    for item in first_malformed_rows:
        print(
            f"  Line {item['line']}: "
            f"{item['field_count']} fields | "
            f"{item['row_preview']}"
        )


# ============================================================
# 5. LOAD RAW DATA
# ============================================================

print()
print("-" * 90)
print("3. LOADING EXISTING RAW DATA")
print("-" * 90)

raw_df = pd.read_csv(
    RAW_PRICE_FILE,
    low_memory=False
)

print("Loaded rows:", f"{len(raw_df):,}")
print("Loaded columns:", len(raw_df.columns))

print()
print("Columns:")
for column in raw_df.columns:
    print("  -", column)


# ============================================================
# 6. SCHEMA CHECK
# ============================================================

print()
print("-" * 90)
print("4. RAW SCHEMA CHECK")
print("-" * 90)

missing_columns = [
    column
    for column in RAW_COLUMNS
    if column not in raw_df.columns
]

unexpected_columns = [
    column
    for column in raw_df.columns
    if column not in RAW_COLUMNS
]

print("Missing expected columns:", missing_columns if missing_columns else "None")
print(
    "Unexpected columns:",
    unexpected_columns if unexpected_columns else "None"
)

schema_ok = (
    not missing_columns
    and not unexpected_columns
)

print("Schema status:", "PASS" if schema_ok else "CHECK REQUIRED")


# ============================================================
# 7. TYPE CONVERSION FOR AUDIT ONLY
# ============================================================

audit_df = raw_df.copy()

audit_df["Date"] = pd.to_datetime(
    audit_df["Date"],
    errors="coerce"
)

for column in [
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
]:
    audit_df[column] = pd.to_numeric(
        audit_df[column],
        errors="coerce"
    )


# ============================================================
# 8. BASIC DATASET SUMMARY
# ============================================================

print()
print("-" * 90)
print("5. BASIC DATASET SUMMARY")
print("-" * 90)

print("Rows:", f"{len(audit_df):,}")
print("Columns:", len(audit_df.columns))

if audit_df["Date"].notna().any():
    print(
        "Date range:",
        audit_df["Date"].min().date(),
        "to",
        audit_df["Date"].max().date()
    )
else:
    print("Date range: unavailable")


# ============================================================
# 9. NULL / INVALID VALUE AUDIT
# ============================================================

print()
print("-" * 90)
print("6. MISSING / INVALID VALUES")
print("-" * 90)

null_counts = audit_df[RAW_COLUMNS].isna().sum()

print("Missing values by important field:")
for column in RAW_COLUMNS:
    count = int(null_counts[column])

    if count:
        percentage = count / len(audit_df) * 100
        print(
            f"  {column}: {count:,} "
            f"({percentage:.3f}%)"
        )
    else:
        print(f"  {column}: 0")


invalid_modal = (
    audit_df["Modal Price"].isna()
    | (audit_df["Modal Price"] <= 0)
)

invalid_minimum = (
    audit_df["Minimum Price"].isna()
    | (audit_df["Minimum Price"] < 0)
)

invalid_maximum = (
    audit_df["Maximum Price"].isna()
    | (audit_df["Maximum Price"] < 0)
)

price_order_problem = (
    audit_df["Minimum Price"].notna()
    & audit_df["Maximum Price"].notna()
    & (
        audit_df["Minimum Price"]
        > audit_df["Maximum Price"]
    )
)

print()
print("Invalid Modal Price rows:", f"{invalid_modal.sum():,}")
print("Invalid Minimum Price rows:", f"{invalid_minimum.sum():,}")
print("Invalid Maximum Price rows:", f"{invalid_maximum.sum():,}")
print(
    "Minimum > Maximum rows:",
    f"{price_order_problem.sum():,}"
)


# ============================================================
# 10. DUPLICATE AUDIT
# ============================================================

print()
print("-" * 90)
print("7. DUPLICATE AUDIT")
print("-" * 90)

available_key_columns = [
    column
    for column in KEY_COLUMNS
    if column in audit_df.columns
]

exact_duplicate_rows = int(
    audit_df.duplicated().sum()
)

key_duplicate_rows = int(
    audit_df.duplicated(
        subset=available_key_columns
    ).sum()
)

print(
    "Exact duplicate rows:",
    f"{exact_duplicate_rows:,}"
)

print(
    "Duplicate forecasting keys:",
    f"{key_duplicate_rows:,}"
)

print("Duplicate key columns:")
for column in available_key_columns:
    print("  -", column)


# ============================================================
# 11. CROP COVERAGE
# ============================================================

print()
print("-" * 90)
print("8. CROP COVERAGE")
print("-" * 90)

crop_summary = (
    audit_df
    .groupby("Crop")
    .agg(
        Observations=("Crop", "size"),
        First_Date=("Date", "min"),
        Last_Date=("Date", "max"),
        Unique_States=("State ID", "nunique"),
        Unique_Markets=("Market ID", "nunique"),
        Unique_Varieties=("Variety", "nunique"),
    )
    .sort_values(
        "Observations",
        ascending=False
    )
)

print(crop_summary.to_string())


# ============================================================
# 12. STATE / MARKET COVERAGE
# ============================================================

print()
print("-" * 90)
print("9. GEOGRAPHIC COVERAGE")
print("-" * 90)

print(
    "Unique states:",
    audit_df["State ID"].nunique()
)

print(
    "Unique markets:",
    audit_df["Market ID"].nunique()
)

print(
    "Unique varieties:",
    audit_df["Variety"].nunique()
)

print()
print("Top 20 states by observations:")

state_summary = (
    audit_df
    .groupby("State")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print(state_summary.to_string())


# ============================================================
# 13. MONTHLY CROP-MONTH COVERAGE
# ============================================================

print()
print("-" * 90)
print("10. MONTHLY CROP-MONTH COVERAGE")
print("-" * 90)

coverage_df = audit_df[
    audit_df["Date"].notna()
].copy()

coverage_df["Year"] = coverage_df["Date"].dt.year
coverage_df["Month"] = coverage_df["Date"].dt.month

coverage_df = (
    coverage_df[
        [
            "Crop",
            "Year",
            "Month",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Crop",
            "Year",
            "Month",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Observed crop-month combinations:",
    f"{len(coverage_df):,}"
)


# ============================================================
# 14. EXPECTED 2015–2025 COVERAGE
# ============================================================

INGESTION_YEARS = range(2015, 2026)

crop_names = sorted(
    audit_df["Crop"]
    .dropna()
    .astype(str)
    .unique()
)

expected_rows = []

for crop in crop_names:
    for year in INGESTION_YEARS:
        for month in range(1, 13):
            expected_rows.append(
                {
                    "Crop": crop,
                    "Year": year,
                    "Month": month,
                }
            )

expected_df = pd.DataFrame(expected_rows)

observed_keys = set(
    zip(
        coverage_df["Crop"],
        coverage_df["Year"],
        coverage_df["Month"],
    )
)

expected_df["Observed"] = [
    (row.Crop, row.Year, row.Month)
    in observed_keys
    for row in expected_df.itertuples()
]

missing_crop_months = (
    expected_df[
        ~expected_df["Observed"]
    ]
    .drop(columns="Observed")
    .reset_index(drop=True)
)

print()
print(
    "Crops found in existing raw data:",
    len(crop_names)
)

print(
    "Expected crop-month combinations "
    f"for those crops, 2015–2025: "
    f"{len(expected_df):,}"
)

print(
    "Observed crop-month combinations:",
    f"{int(expected_df['Observed'].sum()):,}"
)

print(
    "Missing crop-month combinations:",
    f"{len(missing_crop_months):,}"
)

if not missing_crop_months.empty:
    print()
    print("Missing crop-months:")
    print(
        missing_crop_months
        .to_string(index=False)
    )


# ============================================================
# 15. MONTHLY DATASET AUDIT
# ============================================================

print()
print("-" * 90)
print("11. EXISTING MONTHLY DATASET")
print("-" * 90)

monthly_df = pd.DataFrame()

if MONTHLY_PRICE_FILE.exists():

    monthly_df = pd.read_csv(
        MONTHLY_PRICE_FILE,
        low_memory=False
    )

    print(
        "Monthly rows:",
        f"{len(monthly_df):,}"
    )

    print(
        "Monthly columns:",
        len(monthly_df.columns)
    )

    print()
    print("Monthly columns:")
    for column in monthly_df.columns:
        print("  -", column)

    if "Date" in monthly_df.columns:

        monthly_df["Date"] = pd.to_datetime(
            monthly_df["Date"],
            errors="coerce"
        )

        print(
            "Monthly date range:",
            monthly_df["Date"].min().date(),
            "to",
            monthly_df["Date"].max().date()
        )

    if "Crop" in monthly_df.columns:

        print(
            "Monthly crops:",
            monthly_df["Crop"].nunique()
        )

    if {
        "Crop",
        "Year",
        "Month",
    }.issubset(monthly_df.columns):

        monthly_duplicate_count = int(
            monthly_df.duplicated(
                subset=[
                    "Crop",
                    "Year",
                    "Month",
                ]
            ).sum()
        )

        print(
            "Monthly duplicate Crop-Year-Month keys:",
            monthly_duplicate_count
        )

    if "Price" in monthly_df.columns:

        monthly_df["Price"] = pd.to_numeric(
            monthly_df["Price"],
            errors="coerce"
        )

        print(
            "Missing monthly Price:",
            int(monthly_df["Price"].isna().sum())
        )

        print(
            "Non-positive monthly Price:",
            int((monthly_df["Price"] <= 0).sum())
        )

else:

    print("Monthly processed file does not exist.")


# ============================================================
# 16. INGESTION LOG AUDIT
# ============================================================

print()
print("-" * 90)
print("12. INGESTION LOG")
print("-" * 90)

if LOG_FILE.exists():

    log_df = pd.read_csv(
        LOG_FILE,
        low_memory=False
    )

    print(
        "Log entries:",
        f"{len(log_df):,}"
    )

    if "Success" in log_df.columns:

        success_mask = (
            log_df["Success"]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )

        print(
            "Successful log entries:",
            f"{int(success_mask.sum()):,}"
        )

        print(
            "Failed log entries:",
            f"{int((~success_mask).sum()):,}"
        )

    if "HTTP_Status" in log_df.columns:

        print()
        print("HTTP status distribution:")

        print(
            log_df["HTTP_Status"]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

    if "Error" in log_df.columns:

        print()
        print("Most common errors:")

        error_counts = (
            log_df["Error"]
            .fillna("")
            .astype(str)
            .replace("", "NO ERROR")
            .value_counts()
            .head(15)
        )

        print(error_counts.to_string())

else:

    print("Ingestion log does not exist.")


# ============================================================
# 17. TARGET CROP COVERAGE CHECK
# ============================================================

target_crops = [
    "Wheat",
    "Rice",
    "Maize",
    "Jowar/Sorghum",
    "Groundnut",
    "Cotton",
    "Banana",
    "Onion",
    "Potato",
    "Lentil",
    "Sweet Potato",
    "Bajra/Pearl Millet",
]

print()
print("-" * 90)
print("13. PROJECT TARGET CROP COVERAGE")
print("-" * 90)

target_rows = []

for crop in target_crops:

    crop_data = audit_df[
        audit_df["Crop"].astype(str) == crop
    ]

    observed_months = (
        crop_data[
            crop_data["Date"].notna()
        ][
            ["Date"]
        ]
        .assign(
            Year=lambda x: x["Date"].dt.year,
            Month=lambda x: x["Date"].dt.month,
        )[
            ["Year", "Month"]
        ]
        .drop_duplicates()
    )

    expected_count = 11 * 12
    observed_count = len(
        observed_months[
            observed_months["Year"].between(
                2015,
                2025
            )
        ]
    )

    target_rows.append(
        {
            "Crop": crop,
            "Observations": len(crop_data),
            "First_Date": (
                crop_data["Date"].min()
                if len(crop_data)
                else pd.NaT
            ),
            "Last_Date": (
                crop_data["Date"].max()
                if len(crop_data)
                else pd.NaT
            ),
            "Observed_2015_2025_Months": observed_count,
            "Expected_2015_2025_Months": expected_count,
            "Missing_2015_2025_Months": (
                expected_count - observed_count
            ),
            "Markets": crop_data["Market ID"].nunique(),
            "States": crop_data["State ID"].nunique(),
        }
    )

target_coverage_df = pd.DataFrame(target_rows)

print(
    target_coverage_df.to_string(
        index=False
    )
)


# ============================================================
# 18. DATA USABILITY SIGNALS
# ============================================================

print()
print("-" * 90)
print("14. DATA USABILITY SIGNALS")
print("-" * 90)

usable_rows = (
    audit_df["Date"].notna()
    & audit_df["Modal Price"].notna()
    & (audit_df["Modal Price"] > 0)
)

print(
    "Rows with valid Date + positive Modal Price:",
    f"{int(usable_rows.sum()):,}"
)

usable_percentage = (
    usable_rows.mean() * 100
    if len(audit_df)
    else 0
)

print(
    "Usable-row percentage:",
    f"{usable_percentage:.3f}%"
)

print()
print("This audit does NOT declare the dataset complete.")
print("It reports the actual coverage currently available.")
print("Missing months should not be artificially filled merely")
print("to make the historical period look complete.")


# ============================================================
# 19. SAVE AUDIT OUTPUTS
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

coverage_report = expected_df.copy()

coverage_report["Status"] = np.where(
    coverage_report["Observed"],
    "Observed",
    "Missing"
)

coverage_report.to_csv(
    COVERAGE_OUTPUT_FILE,
    index=False
)


# Write a concise text report without modifying source data.
with open(
    AUDIT_OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "FOOD PRICE FORECASTING — DATA AUDIT REPORT\n"
    )
    report.write("=" * 70 + "\n\n")

    report.write(
        f"Raw CSV rows scanned: {total_csv_rows:,}\n"
    )
    report.write(
        f"Malformed CSV rows: {malformed_rows:,}\n"
    )
    report.write(
        f"Loaded raw observations: {len(audit_df):,}\n"
    )
    report.write(
        f"Valid Date + positive Modal Price rows: "
        f"{int(usable_rows.sum()):,}\n"
    )
    report.write(
        f"Exact duplicate rows: {exact_duplicate_rows:,}\n"
    )
    report.write(
        f"Duplicate forecasting keys: {key_duplicate_rows:,}\n"
    )
    report.write(
        f"Unique crops: {audit_df['Crop'].nunique()}\n"
    )
    report.write(
        f"Unique markets: {audit_df['Market ID'].nunique()}\n"
    )
    report.write(
        f"Unique states: {audit_df['State ID'].nunique()}\n"
    )

    if audit_df["Date"].notna().any():
        report.write(
            f"Date range: "
            f"{audit_df['Date'].min().date()} "
            f"to "
            f"{audit_df['Date'].max().date()}\n"
        )

    report.write(
        f"Missing 2015–2025 crop-months: "
        f"{len(missing_crop_months):,}\n"
    )

    report.write(
        f"Monthly dataset rows: "
        f"{len(monthly_df):,}\n"
    )

print()
print("=" * 90)
print("DATA AUDIT COMPLETE")
print("=" * 90)

print()
print("Audit report written to:")
print(AUDIT_OUTPUT_FILE)

print()
print("Crop-month coverage written to:")
print(COVERAGE_OUTPUT_FILE)

print()
print("IMPORTANT:")
print("No AGMARKNET request was made.")
print("No source data was modified.")
print("No missing values were artificially filled.")
print()
print("Next step: review this output before modifying Notebook 02.")


FOOD PRICE FORECASTING — EXISTING DATA AUDIT

Project root: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting
Raw file: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations.csv
Monthly file: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\monthly_crop_prices.csv
Ingestion log: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs\agmarknet_ingestion_log.csv

------------------------------------------------------------------------------------------
1. FILE EXISTENCE
------------------------------------------------------------------------------------------
Raw AGMARKNET file: EXISTS | 50.58 MB
Monthly processed file: EXISTS | 0.09 MB
Ingestion log: EXISTS | 0.81 MB

------------------------------------------------------------------------------------------
2. RAW CSV STRUCTURE
-----------------------------------------------------------------------------------

In [ ]:
import sys
import time
import csv
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Food Price Forecasting")
print("FAST data ingestion pipeline initialized.")

# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    OUTPUT_DIR,
    MODEL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\nProject directories prepared.")
print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Output directory:", OUTPUT_DIR)
print("Model directory:", MODEL_DIR)

# ============================================================
# 2. FAOSTAT — OPTIONAL / NON-BLOCKING
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from config import FAOSTAT_API_TOKEN
except Exception:
    FAOSTAT_API_TOKEN = ""

FAOSTAT_BASE_URL = "https://faostatservices.fao.org/api/v1/en"

FAOSTAT_DOMAINS = {
    "PP": "Producer Prices",
    "QCL": "Crops and livestock products",
    "QI": "Production Indices",
    "QV": "Value of Agricultural Production",
    "CP": "Consumer Price Indices",
    "TCL": "Crops and livestock products"
}

print("\nFAOSTAT configuration loaded.")
print("Known project domains:", ", ".join(FAOSTAT_DOMAINS.keys()))


def faostat_get(path, params=None, timeout=10):
    url = f"{FAOSTAT_BASE_URL}/{path.lstrip('/')}"

    try:
        response = requests.get(
            url,
            headers={
                "Authorization": f"Bearer {FAOSTAT_API_TOKEN}",
                "Accept": "application/json"
            },
            params=params,
            timeout=timeout
        )

        if response.status_code == 403:
            print(f"\nFAOSTAT 403 skipped: {url}")
            return None

        if not response.ok:
            print(
                f"\nFAOSTAT request skipped: "
                f"HTTP {response.status_code}"
            )
            return None

        try:
            return response.json()
        except ValueError:
            return None

    except requests.RequestException as error:
        print("\nFAOSTAT request skipped:", error)
        return None


producer_prices_dimensions = faostat_get(
    "dimensions/PP"
)

if producer_prices_dimensions:
    print(
        "\nFAOSTAT Producer Prices dimensions available."
    )
else:
    print(
        "\nFAOSTAT Producer Prices dimensions unavailable; "
        "continuing with AGMARKNET."
    )

print(
    "FAOSTAT investigation will not block the project."
)

# ============================================================
# 3. AGMARKNET CONFIGURATION
# ============================================================

AGMARKNET_BASE_URL = (
    "https://api.agmarknet.gov.in/v1"
)

agmarknet_headers = {
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://agmarknet.gov.in",
    "Referer": "https://agmarknet.gov.in/",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    )
}

FILTERS_URL = (
    f"{AGMARKNET_BASE_URL}/daily-price-arrival/filters"
)

PRICE_URL = (
    f"{AGMARKNET_BASE_URL}/prices-and-arrivals/"
    "market-report/specific"
)

print("\nTesting AGMARKNET connectivity...")

response = requests.get(
    FILTERS_URL,
    headers=agmarknet_headers,
    timeout=12
)

response.raise_for_status()

agmarknet_data = response.json().get(
    "data",
    {}
)

print(
    "AGMARKNET status:",
    response.status_code
)

print(
    "Response type:",
    response.headers.get("content-type")
)

print(
    "AGMARKNET API is reachable."
)

print("\nAGMARKNET filter groups:")

for key, value in agmarknet_data.items():

    if isinstance(value, list):
        print(
            f" - {key}: {len(value)} records"
        )

    elif isinstance(value, dict):
        print(
            f" - {key}: dictionary"
        )

# ============================================================
# 4. COMMODITY INSPECTION
# ============================================================

cmdt_records = agmarknet_data.get(
    "cmdt_data",
    []
)

target_crop_keywords = [
    "wheat",
    "rice",
    "maize",
    "corn",
    "potato",
    "tomato",
    "onion",
    "sugarcane",
    "cotton",
    "groundnut",
    "lentil",
    "pigeon pea",
    "pigeonpea",
    "tur",
    "sorghum",
    "jowar",
    "millet",
    "bajra",
    "barley",
    "banana",
    "sweet potato"
]

commodity_matches = []

for record in cmdt_records:

    commodity_name = str(
        record.get(
            "cmdt_name",
            ""
        )
    )

    if any(
        keyword.lower()
        in commodity_name.lower()
        for keyword in target_crop_keywords
    ):

        commodity_matches.append({
            "Commodity ID":
                record.get("cmdt_id"),

            "Commodity":
                commodity_name,

            "Group ID":
                record.get("cmdt_group_id")
        })


commodity_matches_df = pd.DataFrame(
    commodity_matches
).drop_duplicates()

print(
    "\nMatching AGMARKNET commodities:",
    len(commodity_matches_df)
)

if not commodity_matches_df.empty:
    print(
        commodity_matches_df.to_string(
            index=False
        )
    )

# ============================================================
# 5. VERIFIED PROJECT CROP SET
# ============================================================

final_agmarknet_crops = {

    "Wheat": {
        "commodity_id": 1,
        "group_id": 1
    },

    "Rice": {
        "commodity_id": 3,
        "group_id": 1
    },

    "Maize": {
        "commodity_id": 4,
        "group_id": 1
    },

    "Jowar/Sorghum": {
        "commodity_id": 5,
        "group_id": 1
    },

    "Groundnut": {
        "commodity_id": 10,
        "group_id": 3
    },

    "Cotton": {
        "commodity_id": 15,
        "group_id": 4
    },

    "Banana": {
        "commodity_id": 19,
        "group_id": 5
    },

    "Onion": {
        "commodity_id": 23,
        "group_id": 6
    },

    "Potato": {
        "commodity_id": 24,
        "group_id": 6
    },

    "Lentil": {
        "commodity_id": 52,
        "group_id": 2
    },

    "Sweet Potato": {
        "commodity_id": 124,
        "group_id": 6
    },

    "Bajra/Pearl Millet": {
        "commodity_id": 28,
        "group_id": 1
    }
}

print("\nFinal AGMARKNET crop set:")

for crop_name, info in final_agmarknet_crops.items():

    print(
        f" - {crop_name}: "
        f"commodity={info['commodity_id']}, "
        f"group={info['group_id']}"
    )

# ============================================================
# ============================================================
# 6. QUICK ENDPOINT TEST
# ============================================================

test_response = requests.get(
    PRICE_URL,
    headers=agmarknet_headers,
    params={
        "date": "2026-09-01",
        "commodityGroupId": 1,
        "commodityId": 1
    },
    timeout=12
)

print(
    "\nAGMARKNET specific price endpoint status:",
    test_response.status_code
)

if test_response.status_code == 429:

    print(
        "WARNING: AGMARKNET is currently rate-limiting requests (HTTP 429)."
    )

    print(
        "The API endpoint is reachable, but the quick test will be skipped."
    )

    test_json = None

elif test_response.ok:

    test_json = test_response.json()

    print(
        "Title:",
        test_json.get("title")
    )

    print(
        "Commodity:",
        test_json.get("commodityName")
    )

    print(
        "States:",
        len(
            test_json.get(
                "states",
                []
            )
        )
    )

else:

    print(
        "WARNING: AGMARKNET quick endpoint test returned HTTP",
        test_response.status_code
    )

    print(
        "Continuing without stopping the ingestion pipeline."
    )

    test_json = None

# ============================================================
# 7. FAST INGESTION SETTINGS
# ============================================================

INGESTION_YEARS = range(
    2015,
    2026
)

PRIMARY_DAY = 1

FALLBACK_DAYS = [
    8,
    15
]

REQUEST_TIMEOUT = 15
MAX_RETRIES = 0
MAX_WORKERS = 1

# Rate-limit handling
RATE_LIMIT_BASE_DELAY = 5
RATE_LIMIT_MAX_DELAY = 60

RAW_PRICE_FILE = (
    RAW_DIR /
    "agmarknet_price_observations.csv"
)

LOG_FILE = (
    OUTPUT_DIR /
    "agmarknet_ingestion_log.csv"
)

MONTHLY_PRICE_FILE = (
    PROCESSED_DIR /
    "monthly_crop_prices.csv"
)

print(
    "\nFAST ingestion configuration:"
)

print(
    "Years:",
    f"{min(INGESTION_YEARS)}–"
    f"{max(INGESTION_YEARS)}"
)

print(
    "Crops:",
    len(final_agmarknet_crops)
)

print(
    "Primary day:",
    PRIMARY_DAY
)

print(
    "Fallback days:",
    FALLBACK_DAYS
)

print(
    "Parallel workers:",
    MAX_WORKERS
)

print(
    "Existing raw file:",
    RAW_PRICE_FILE
)

# ============================================================
# 8. LOAD EXISTING PROGRESS
# ============================================================

if RAW_PRICE_FILE.exists():

    try:
        raw_existing_df = pd.read_csv(
            RAW_PRICE_FILE
        )
    except Exception:
        raw_existing_df = pd.DataFrame()

else:
    raw_existing_df = pd.DataFrame()


if LOG_FILE.exists():

    try:
        ingestion_log_df = pd.read_csv(
            LOG_FILE
        )
    except Exception:
        ingestion_log_df = pd.DataFrame()

else:
    ingestion_log_df = pd.DataFrame()


print(
    "\nExisting raw observations:",
    f"{len(raw_existing_df):,}"
)

print(
    "Existing log entries:",
    f"{len(ingestion_log_df):,}"
)

# ============================================================
# 9. DETERMINE ALREADY COMPLETED MONTHS
# ============================================================

completed_month_keys = set()

if not ingestion_log_df.empty:

    required_log_columns = {
        "Crop",
        "Year",
        "Month",
        "Success",
        "Observations"
    }

    if required_log_columns.issubset(
        ingestion_log_df.columns
    ):

        temp_log = ingestion_log_df.copy()

        temp_log["Year_num"] = pd.to_numeric(
            temp_log["Year"],
            errors="coerce"
        )

        temp_log["Month_num"] = pd.to_numeric(
            temp_log["Month"],
            errors="coerce"
        )

        temp_log["Success_bool"] = (
            temp_log["Success"]
            .astype(str)
            .str.lower()
            .isin(
                [
                    "true",
                    "1",
                    "yes"
                ]
            )
        )

        temp_log["Obs_num"] = (
            pd.to_numeric(
                temp_log["Observations"],
                errors="coerce"
            )
            .fillna(0)
        )

        valid_log = temp_log[
            temp_log["Success_bool"]
            &
            (
                temp_log["Obs_num"] > 0
            )
        ]

        for row in (
            valid_log[
                [
                    "Crop",
                    "Year_num",
                    "Month_num"
                ]
            ]
            .drop_duplicates()
            .itertuples(
                index=False
            )
        ):

            completed_month_keys.add(
                (
                    str(row.Crop),
                    int(row.Year_num),
                    int(row.Month_num)
                )
            )


print(
    "Completed crop-months already available:",
    len(completed_month_keys)
)

# ============================================================
# 10. REQUEST / FLATTENING FUNCTION
# ============================================================

def fetch_agmarknet_day(
    crop_name,
    crop_info,
    date_string
):

    params = {
        "date": date_string,
        "commodityGroupId": crop_info["group_id"],
        "commodityId": crop_info["commodity_id"]
    }

    last_error = ""
    last_status = None

    for attempt in range(
        MAX_RETRIES + 1
    ):

        try:

            response = requests.get(
                PRICE_URL,
                headers=agmarknet_headers,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            last_status = response.status_code

            # ----------------------------------------------------
            # HTTP 429 — AGMARKNET RATE LIMIT
            # ----------------------------------------------------

            if response.status_code == 429:

                return {
                    "success": False,
                    "rows": [],
                    "status": 429,
                    "error": (
                        "HTTP 429: AGMARKNET rate limit"
                    )
                }

            # ----------------------------------------------------
            # SUCCESS
            # ----------------------------------------------------

            if response.ok:

                data = response.json()

                rows = []

                for state in data.get(
                    "states",
                    []
                ):

                    state_name = state.get(
                        "stateName",
                        ""
                    )

                    state_id = state.get(
                        "stateId",
                        ""
                    )

                    for market in state.get(
                        "markets",
                        []
                    ):

                        market_name = market.get(
                            "marketName",
                            ""
                        )

                        market_id = market.get(
                            "marketId",
                            ""
                        )

                        for variety in market.get(
                            "varieties",
                            []
                        ):

                            variety_name = variety.get(
                                "varietyName",
                                ""
                            )

                            grade_name = variety.get(
                                "gradeName",
                                ""
                            )

                            minimum_price = variety.get(
                                "minimumPrice"
                            )

                            maximum_price = variety.get(
                                "maximumPrice"
                            )

                            modal_price = variety.get(
                                "modalPrice"
                            )

                            unit = variety.get(
                                "unit",
                                ""
                            )

                            if modal_price in (
                                None,
                                "",
                                "-"
                            ):
                                continue

                            rows.append(
                                {
                                    "Date": date_string,
                                    "Crop": crop_name,
                                    "Commodity ID":
                                        crop_info["commodity_id"],
                                    "Commodity Group ID":
                                        crop_info["group_id"],
                                    "State": state_name,
                                    "State ID": state_id,
                                    "Market": market_name,
                                    "Market ID": market_id,
                                    "Variety": variety_name,
                                    "Grade": grade_name,
                                    "Minimum Price":
                                        minimum_price,
                                    "Maximum Price":
                                        maximum_price,
                                    "Modal Price":
                                        modal_price,
                                    "Unit": unit
                                }
                            )

                if rows:

                    return {
                        "success": True,
                        "rows": rows,
                        "status": last_status,
                        "error": ""
                    }

                last_error = (
                    "HTTP 200 but no usable price rows returned"
                )

            # ----------------------------------------------------
            # OTHER HTTP ERRORS
            # ----------------------------------------------------

            else:

                last_error = (
                    f"HTTP {response.status_code}"
                )

                if attempt < MAX_RETRIES:

                    delay = min(
                        2 ** attempt,
                        15
                    )

                    time.sleep(delay)

                    continue

                break

        except requests.exceptions.RequestException as exc:

            last_error = str(exc)

            if attempt < MAX_RETRIES:

                delay = min(
                    2 ** attempt,
                    15
                )

                time.sleep(delay)

                continue

            break

        except Exception as exc:

            last_error = str(exc)

            break

    return {
        "success": False,
        "rows": [],
        "status": last_status,
        "error": last_error
    }

# ============================================================
# 11. BUILD ONLY MISSING REQUESTS
# ============================================================

tasks = []

for crop_name, crop_info in (
    final_agmarknet_crops.items()
):

    for year in INGESTION_YEARS:

        for month in range(
            1,
            13
        ):

            if (
                crop_name,
                year,
                month
            ) in completed_month_keys:

                continue

            tasks.append(
                (
                    crop_name,
                    crop_info,
                    year,
                    month
                )
            )


total_possible = (
    len(final_agmarknet_crops)
    *
    len(list(INGESTION_YEARS))
    *
    12
)

print(
    "\n" + "=" * 90
)

print(
    "FAST RESUMABLE AGMARKNET INGESTION"
)

print(
    "=" * 90
)

print(
    "Total crop-months in project:",
    total_possible
)

print(
    "Already completed:",
    len(completed_month_keys)
)

print(
    "Crop-months still requiring work:",
    len(tasks)
)

if not tasks:

    print(
        "\nAll crop-months already have "
        "usable observations."
    )

else:

    print(
        "\nMaximum normal requests for this run:",
        len(tasks),
        "(plus fallback requests only when necessary)"
    )

# ============================================================
# 12. FAST MONTH FETCH
# ============================================================

def fetch_month(task):

    crop_name, crop_info, year, month = task

    candidate_days = (
        [PRIMARY_DAY]
        +
        FALLBACK_DAYS
    )

    attempts = []

    for day in candidate_days:

        try:

            date_obj = pd.Timestamp(
                year=year,
                month=month,
                day=day
            )

        except ValueError:

            continue

        date_string = (
            date_obj.strftime(
                "%Y-%m-%d"
            )
        )

        result = fetch_agmarknet_day(
            crop_name,
            crop_info,
            date_string
        )

        records = result.get(
            "rows",
            []
        )

        metadata = {
            "Crop":
                crop_name,

            "Year":
                year,

            "Month":
                month,

            "Date":
                date_string,

            "HTTP_Status":
                result.get(
                    "status"
                ),

            "Success":
                result.get(
                    "success",
                    False
                ),

            "States":
                0,

            "Markets":
                0,

            "Observations":
                len(records),

            "Error":
                result.get(
                    "error",
                    ""
                )
        }

        attempts.append(
            (
                records,
                metadata
            )
        )

        if (
            metadata["Success"]
            and records
        ):

            return (
                crop_name,
                year,
                month,
                records,
                metadata,
                attempts
            )

        # If AGMARKNET is rate-limiting us, do not try
        # fallback dates for the same crop-month.
        if metadata["HTTP_Status"] == 429:
            break

    if attempts:

        records, metadata = attempts[-1]

        return (
            crop_name,
            year,
            month,
            records,
            metadata,
            attempts
        )

    return (
        crop_name,
        year,
        month,
        [],
        {
            "Crop":
                crop_name,

            "Year":
                year,

            "Month":
                month,

            "Date":
                "",

            "HTTP_Status":
                None,

            "Success":
                False,

            "States":
                0,

            "Markets":
                0,

            "Observations":
                0,

            "Error":
                "No valid candidate date"
        },
        []
    )

# ============================================================
# 13. SAVE HELPER
# ============================================================

def append_df(
    df,
    filepath
):

    if (
        df is None
        or df.empty
    ):

        return

    filepath.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    header = (
        not filepath.exists()
    )

    df.to_csv(
        filepath,
        mode="a",
        header=header,
        index=False
    )

# ============================================================
# 14. PARALLEL INGESTION
# ============================================================

new_raw_rows = 0
completed_this_run = 0
failed_this_run = 0
successful_dates_this_run = 0

try:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        future_map = {
            executor.submit(
                fetch_month,
                task
            ): task

            for task in tasks
        }

        for completed_count, future in enumerate(
            as_completed(
                future_map
            ),
            start=1
        ):

            task = future_map[future]

            try:

                (
                    crop_name,
                    year,
                    month,
                    records,
                    metadata,
                    attempts
                ) = future.result()

            except Exception as error:

                crop_name, crop_info, year, month = task

                records = []

                metadata = {

                    "Crop":
                        crop_name,

                    "Year":
                        year,

                    "Month":
                        month,

                    "Date":
                        "",

                    "HTTP_Status":
                        None,

                    "Success":
                        False,

                    "States":
                        0,

                    "Markets":
                        0,

                    "Observations":
                        0,

                    "Error":
                        str(error)

                }

                attempts = []

            for (
                _,
                attempt_metadata
            ) in attempts:

                append_df(
                    pd.DataFrame(
                        [attempt_metadata]
                    ),
                    LOG_FILE
                )

            if (
                metadata.get(
                    "Success"
                )
                and
                records
            ):

                raw_day_df = pd.DataFrame(
                    records
                )

                for column in [
                    "Minimum Price",
                    "Maximum Price",
                    "Modal Price",
                    "Arrival"
                ]:

                    if column in raw_day_df.columns:

                        raw_day_df[column] = (
                            pd.to_numeric(
                                raw_day_df[column],
                                errors="coerce"
                            )
                        )

                raw_day_df = (
                    raw_day_df.drop_duplicates(
                        subset=[
                            "Date",
                            "Crop",
                            "State ID",
                            "Market ID",
                            "Variety",
                            "Grade"
                        ]
                    )
                )

                append_df(
                    raw_day_df,
                    RAW_PRICE_FILE
                )

                new_raw_rows += len(
                    raw_day_df
                )

                completed_this_run += 1

                successful_dates_this_run += 1

            else:

                failed_this_run += 1

            if (
                completed_count % 10 == 0
                or
                completed_count == len(tasks)
            ):

                print(
                    f"Progress: "
                    f"{completed_count:,}/"
                    f"{len(tasks):,} "
                    f"| successful: "
                    f"{completed_this_run:,} "
                    f"| failed: "
                    f"{failed_this_run:,} "
                    f"| new raw rows: "
                    f"{new_raw_rows:,}"
                )

except KeyboardInterrupt:

    print(
        "\nKeyboardInterrupt received."
    )

    print(
        "Completed futures have already been saved."
    )

    print(
        "Rerun this cell to resume."
    )


print(
    "\n" + "=" * 90
)

print(
    "FAST INGESTION STAGE COMPLETE"
)

print(
    "=" * 90
)

print(
    "Successful crop-months this run:",
    completed_this_run
)

print(
    "Failed crop-months this run:",
    failed_this_run
)

print(
    "New raw rows written:",
    f"{new_raw_rows:,}"
)

# ============================================================
# 15. LOAD, CLEAN AND DEDUPLICATE RAW DATA
# ============================================================

# Repair any legacy 16-field rows from earlier ingestion runs.
# The current ingestion schema is strictly 14 fields.
if RAW_PRICE_FILE.exists():

    RAW_COLUMNS = [
        "Date",
        "Crop",
        "Commodity ID",
        "Commodity Group ID",
        "State",
        "State ID",
        "Market",
        "Market ID",
        "Variety",
        "Grade",
        "Minimum Price",
        "Maximum Price",
        "Modal Price",
        "Unit",
    ]

    RAW_REPAIRED_FILE = RAW_PRICE_FILE.with_suffix(".repaired.csv")

    with open(
        RAW_PRICE_FILE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as source, open(
        RAW_REPAIRED_FILE,
        "w",
        encoding="utf-8",
        newline=""
    ) as target:

        reader = csv.reader(source)
        writer = csv.writer(target)

        try:
            next(reader)
        except StopIteration:
            pass

        writer.writerow(RAW_COLUMNS)

        for row in reader:

            if len(row) == 14:
                writer.writerow(row)

            elif len(row) == 16:
                # Earlier runs stored Arrival and Arrival Unit
                # after the 14 price fields. Those two fields are
                # not part of the forecasting raw schema.
                writer.writerow(row[:14])

    RAW_REPAIRED_FILE.replace(RAW_PRICE_FILE)

if RAW_PRICE_FILE.exists():

    raw_price_df = pd.read_csv(
        RAW_PRICE_FILE
    )

else:

    raw_price_df = pd.DataFrame()


print(
    "\nRaw AGMARKNET observations loaded:",
    f"{len(raw_price_df):,}"
)

if not raw_price_df.empty:

    raw_price_df["Date"] = pd.to_datetime(
        raw_price_df["Date"],
        errors="coerce"
    )

    for column in [
        "Modal Price",
        "Minimum Price",
        "Maximum Price",
        "Arrival"
    ]:

        if column in raw_price_df.columns:

            raw_price_df[column] = (
                pd.to_numeric(
                    raw_price_df[column],
                    errors="coerce"
                )
            )

    raw_price_df = raw_price_df[
        raw_price_df["Date"].notna()
        &
        raw_price_df["Modal Price"].notna()
        &
        (
            raw_price_df["Modal Price"] > 0
        )
    ].copy()

    raw_price_df = (
        raw_price_df.drop_duplicates(
            subset=[
                "Date",
                "Crop",
                "State ID",
                "Market ID",
                "Variety",
                "Grade"
            ]
        )
        .reset_index(drop=True)
    )

    raw_price_df["Year"] = (
        raw_price_df["Date"].dt.year
    )

    raw_price_df["Month"] = (
        raw_price_df["Date"].dt.month
    )

    raw_price_df.to_csv(
        RAW_PRICE_FILE,
        index=False
    )

    print(
        "Usable raw observations:",
        f"{len(raw_price_df):,}"
    )

# ============================================================
# 16. MONTHLY NATIONAL PRICE DATASET
# ============================================================

if raw_price_df.empty:

    monthly_price_df = pd.DataFrame()

else:

    monthly_price_df = (
        raw_price_df
        .groupby(
            [
                "Crop",
                "Year",
                "Month"
            ],
            as_index=False
        )
        .agg(
            Price=(
                "Modal Price",
                "median"
            ),

            Mean_Price=(
                "Modal Price",
                "mean"
            ),

            Min_Price=(
                "Modal Price",
                "min"
            ),

            Max_Price=(
                "Modal Price",
                "max"
            ),

            Price_Observations=(
                "Modal Price",
                "count"
            ),

            Markets=(
                "Market ID",
                "nunique"
            ),

            States=(
                "State ID",
                "nunique"
            )
        )
        .sort_values(
            [
                "Crop",
                "Year",
                "Month"
            ]
        )
        .reset_index(drop=True)
    )

    monthly_price_df["Date"] = pd.to_datetime(
        monthly_price_df[
            [
                "Year",
                "Month"
            ]
        ].assign(
            Day=1
        )
    )

    monthly_price_df = monthly_price_df[
        [
            "Date",
            "Year",
            "Month",
            "Crop",
            "Price",
            "Mean_Price",
            "Min_Price",
            "Max_Price",
            "Price_Observations",
            "Markets",
            "States"
        ]
    ]

    monthly_price_df.to_csv(
        MONTHLY_PRICE_FILE,
        index=False
    )

# ============================================================
# 17. FINAL DATA QUALITY CHECK
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "FINAL DATA QUALITY CHECK"
)

print(
    "=" * 90
)

if raw_price_df.empty:

    print(
        "No raw observations available."
    )

else:

    duplicate_key_count = (
        raw_price_df.duplicated(
            subset=[
                "Date",
                "Crop",
                "State ID",
                "Market ID",
                "Variety",
                "Grade"
            ]
        ).sum()
    )

    print(
        "Raw observations:",
        f"{len(raw_price_df):,}"
    )

    print(
        "Exact duplicate keys:",
        duplicate_key_count
    )

    print(
        "Unique crops:",
        raw_price_df["Crop"].nunique()
    )

    print(
        "Unique markets:",
        raw_price_df["Market ID"].nunique()
    )

    print(
        "Unique states:",
        raw_price_df["State ID"].nunique()
    )

    print(
        "Date range:",
        raw_price_df["Date"].min().date(),
        "to",
        raw_price_df["Date"].max().date()
    )

if monthly_price_df.empty:

    print(
        "\nMonthly dataset is empty."
    )

else:

    monthly_duplicate_count = (
        monthly_price_df.duplicated(
            subset=[
                "Crop",
                "Year",
                "Month"
            ]
        ).sum()
    )

    print(
        "\nMonthly rows:",
        f"{len(monthly_price_df):,}"
    )

    print(
        "Monthly duplicate "
        "Crop-Year-Month keys:",
        monthly_duplicate_count
    )

    print(
        "Crops represented:",
        monthly_price_df["Crop"].nunique()
    )

    print(
        "Monthly date range:",
        monthly_price_df["Date"].min().date(),
        "to",
        monthly_price_df["Date"].max().date()
    )

    print(
        "\nRows per crop:"
    )

    print(
        monthly_price_df
        .groupby("Crop")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )

    print(
        "\nSample:"
    )

    print(
        monthly_price_df
        .head(20)
        .to_string(index=False)
    )

# ============================================================
# 18. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "FOOD PRICE FORECASTING — DATA INGESTION STATUS"
)

print(
    "=" * 90
)

print(
    "Primary price source:",
    "Government of India AGMARKNET"
)

print(
    "Forecasting price field:",
    "Modal Price (Rs./Quintal)"
)

print(
    "Raw granularity:",
    "Date + Crop + State + Market + Variety + Grade"
)

print(
    "Monthly target:",
    "National median market Modal Price"
)

print(
    "FAOSTAT:",
    "Optional enrichment source; "
    "403 responses do not block ingestion."
)

print(
    "\nRaw dataset:",
    RAW_PRICE_FILE
)

print(
    "Monthly forecasting dataset:",
    MONTHLY_PRICE_FILE
)

print(
    "\nNext stages:"
    "\n1. Feature engineering"
    "\n2. Exploratory analysis"
    "\n3. Forecasting models"
    "\n4. Model evaluation"
)

Food Price Forecasting
FAST data ingestion pipeline initialized.

Project directories prepared.
Project root: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting
Raw data directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw
Processed data directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed
Output directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs
Model directory: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\models

FAOSTAT configuration loaded.
Known project domains: PP, QCL, QI, QV, CP, TCL

FAOSTAT 403 skipped: https://faostatservices.fao.org/api/v1/en/dimensions/PP

FAOSTAT Producer Prices dimensions unavailable; continuing with AGMARKNET.
FAOSTAT investigation will not block the project.

Testing AGMARKNET connectivity...
AGMARKNET status: 200
Response type: application/json
AGMARKNET API is reachable.

AGMARKNET filter groups:

In [ ]:
from pathlib import Path
import csv
import pandas as pd

RAW_PRICE_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations.csv"

print("=" * 90)
print("RAW CSV VALIDATION")
print("=" * 90)

print(f"File: {RAW_PRICE_FILE}")
print(f"Exists: {RAW_PRICE_FILE.exists()}")

if not RAW_PRICE_FILE.exists():
    raise FileNotFoundError(f"Raw price file not found: {RAW_PRICE_FILE}")

file_size_mb = RAW_PRICE_FILE.stat().st_size / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# ------------------------------------------------------------
# Inspect the CSV structure safely
# ------------------------------------------------------------

with open(
    RAW_PRICE_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:
    reader = csv.reader(f)

    header = next(reader)
    expected_fields = len(header)

    print(f"\nExpected columns: {expected_fields}")
    print("Columns:")
    for i, column in enumerate(header, start=1):
        print(f"  {i}. {column}")

    malformed_rows = []
    total_rows = 0

    for line_number, row in enumerate(reader, start=2):
        total_rows += 1

        if len(row) != expected_fields:
            malformed_rows.append(
                {
                    "line_number": line_number,
                    "field_count": len(row),
                    "expected": expected_fields,
                    "row": row
                }
            )

print("\n" + "-" * 90)
print(f"Total data rows scanned: {total_rows:,}")
print(f"Malformed rows found: {len(malformed_rows):,}")

if malformed_rows:
    print("\nFirst malformed rows:")

    for bad in malformed_rows[:10]:
        print(
            f"\nLine {bad['line_number']}: "
            f"found {bad['field_count']} fields, "
            f"expected {bad['expected']} fields"
        )

        print(bad["row"])

else:
    print("\nNo malformed rows found.")

print("\nRaw CSV validation complete.")

In [ ]:
from pathlib import Path
import csv
import pandas as pd

RAW_PRICE_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations.csv"
CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

print("=" * 90)
print("NORMALIZING AGMARKNET RAW PRICE DATA")
print("=" * 90)

expected_columns = [
    "Date",
    "Crop",
    "Commodity ID",
    "Commodity Group ID",
    "State",
    "State ID",
    "Market",
    "Market ID",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit",
]

rows_written = 0
rows_14 = 0
rows_16 = 0
rows_other = 0

with open(
    RAW_PRICE_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as source, open(
    CLEAN_RAW_FILE,
    "w",
    encoding="utf-8",
    newline=""
) as target:

    reader = csv.reader(source)
    writer = csv.writer(target)

    header = next(reader)

    writer.writerow(expected_columns)

    for line_number, row in enumerate(reader, start=2):

        if len(row) == 14:
            clean_row = row
            rows_14 += 1

        elif len(row) == 16:
            # Keep the 14 price-related fields.
            # Fields 15 and 16 are arrivals and arrivals unit.
            clean_row = row[:14]
            rows_16 += 1

        else:
            rows_other += 1
            continue

        writer.writerow(clean_row)
        rows_written += 1

print("\nNormalization complete.")
print("-" * 90)
print(f"Original 14-field rows : {rows_14:,}")
print(f"16-field rows          : {rows_16:,}")
print(f"Other/malformed rows   : {rows_other:,}")
print(f"Clean rows written     : {rows_written:,}")

clean_size_mb = CLEAN_RAW_FILE.stat().st_size / (1024 * 1024)

print(f"\nClean file:")
print(CLEAN_RAW_FILE)
print(f"Clean file size: {clean_size_mb:.2f} MB")

print("\nPrice columns preserved:")
for column in expected_columns:
    print(f" - {column}")

print("\nThe original raw CSV has NOT been modified.")
print("The normalized file is ready for validation.")

In [ ]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

print("=" * 90)
print("VALIDATING CLEAN AGMARKNET PRICE DATA")
print("=" * 90)

df_clean = pd.read_csv(CLEAN_RAW_FILE)

print(f"\nRows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

print("\nColumn names:")
for column in df_clean.columns:
    print(f" - {column}")

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
missing = df_clean.isna().sum()
print(missing[missing > 0])

print("\nUnique crops:")
for crop in sorted(df_clean["Crop"].dropna().unique()):
    count = (df_clean["Crop"] == crop).sum()
    print(f" - {crop}: {count:,}")

print("\nDate coverage:")
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce")

print(f"Minimum date: {df_clean['Date'].min()}")
print(f"Maximum date: {df_clean['Date'].max()}")

print("\nPrice validation:")

for column in ["Minimum Price", "Maximum Price", "Modal Price"]:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    print(
        f"{column}: "
        f"missing={df_clean[column].isna().sum():,}, "
        f"negative={(df_clean[column] < 0).sum():,}, "
        f"zero={(df_clean[column] == 0).sum():,}"
    )

print("\nModal price summary:")
print(df_clean["Modal Price"].describe())

print("\nDuplicate check:")

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

duplicate_count = df_clean.duplicated(
    subset=duplicate_columns
).sum()

print(
    f"Duplicate observations using "
    f"Date + Crop + State + Market + Variety + Grade: "
    f"{duplicate_count:,}"
)

print("\nValidation complete.")

In [ ]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = Path(PROJECT_ROOT) / "data" / "raw" / "agmarknet_price_observations_clean.csv"

df_clean = pd.read_csv(CLEAN_RAW_FILE)

print("=" * 90)
print("INVESTIGATING DUPLICATE AGMARKNET OBSERVATIONS")
print("=" * 90)

# ------------------------------------------------------------
# Duplicate key
# ------------------------------------------------------------

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

duplicate_mask = df_clean.duplicated(
    subset=duplicate_columns,
    keep=False
)

duplicates = df_clean.loc[
    duplicate_mask
].copy()

print(f"\nTotal rows: {len(df_clean):,}")
print(f"Rows belonging to duplicate groups: {len(duplicates):,}")

# ------------------------------------------------------------
# Number of duplicate groups
# ------------------------------------------------------------

duplicate_groups = (
    duplicates
    .groupby(duplicate_columns, dropna=False)
    .size()
    .reset_index(name="count")
)

print(f"Duplicate groups: {len(duplicate_groups):,}")

print("\nDuplicate group size distribution:")
print(
    duplicate_groups["count"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Duplicates by crop
# ------------------------------------------------------------

print("\nDuplicate rows by crop:")
print(
    duplicates["Crop"]
    .value_counts()
)

# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("EXAMPLES OF DUPLICATE GROUPS")
print("-" * 90)

for _, group in duplicate_groups.head(10).iterrows():

    mask = pd.Series(True, index=df_clean.index)

    for column in duplicate_columns:
        mask &= df_clean[column].astype(str).eq(
            str(group[column])
        )

    example = df_clean.loc[
        mask,
        duplicate_columns + [
            "Minimum Price",
            "Maximum Price",
            "Modal Price",
            "Unit"
        ]
    ]

    print("\nDuplicate group:")
    print(example.to_string(index=False))

print("\nDuplicate investigation complete.")

In [ ]:
import pandas as pd
from pathlib import Path

CLEAN_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_clean.csv"
)

MASTER_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_master.csv"
)

print("=" * 90)
print("CREATING MASTER AGMARKNET PRICE DATASET")
print("=" * 90)

df = pd.read_csv(CLEAN_RAW_FILE)

print(f"\nRows before deduplication: {len(df):,}")

duplicate_columns = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

# Remove exact duplicate observations while keeping the first copy.
df_master = df.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
).copy()

duplicates_removed = len(df) - len(df_master)

print(f"Duplicate rows removed: {duplicates_removed:,}")
print(f"Rows after deduplication: {len(df_master):,}")

# ------------------------------------------------------------
# Save master dataset
# ------------------------------------------------------------

df_master.to_csv(
    MASTER_RAW_FILE,
    index=False
)

print(f"\nMaster dataset saved to:")
print(MASTER_RAW_FILE)

# ------------------------------------------------------------
# Final duplicate verification
# ------------------------------------------------------------

remaining_duplicates = df_master.duplicated(
    subset=duplicate_columns
).sum()

print("\nFinal duplicate check:")
print(f"Remaining duplicate observations: {remaining_duplicates:,}")

# ------------------------------------------------------------
# Crop coverage
# ------------------------------------------------------------

print("\nCrop coverage:")
crop_summary = (
    df_master
    .groupby("Crop")
    .agg(
        observations=("Crop", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        markets=("Market ID", "nunique"),
        states=("State ID", "nunique")
    )
    .sort_values("observations", ascending=False)
)

print(crop_summary.to_string())

print("\nMaster dataset creation complete.")

In [ ]:
import pandas as pd
from pathlib import Path

MASTER_RAW_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "raw"
    / "agmarknet_price_observations_master.csv"
)

MONTHLY_PRICE_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "monthly_crop_prices.csv"
)

print("=" * 90)
print("BUILDING MONTHLY CROP PRICE DATASET")
print("=" * 90)

# ------------------------------------------------------------
# Load master dataset
# ------------------------------------------------------------

df = pd.read_csv(MASTER_RAW_FILE)

print(f"\nMaster observations loaded: {len(df):,}")

# ------------------------------------------------------------
# Prepare dates and prices
# ------------------------------------------------------------

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Modal Price"] = pd.to_numeric(
    df["Modal Price"],
    errors="coerce"
)

# Remove rows where the date or modal price cannot be used.
df = df.dropna(
    subset=["Date", "Modal Price"]
).copy()

# ------------------------------------------------------------
# Remove zero modal prices
# ------------------------------------------------------------

zero_prices = (df["Modal Price"] <= 0).sum()

print(f"Non-positive modal prices removed: {zero_prices:,}")

df = df[
    df["Modal Price"] > 0
].copy()

# ------------------------------------------------------------
# Create monthly period
# ------------------------------------------------------------

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month Start"] = df["Date"].dt.to_period("M").dt.to_timestamp()

# ------------------------------------------------------------
# Aggregate to national monthly level
# ------------------------------------------------------------

monthly = (
    df
    .groupby(
        ["Crop", "Year", "Month", "Month Start"],
        as_index=False
    )
    .agg(
        Median_Price=("Modal Price", "median"),
        Mean_Price=("Modal Price", "mean"),
        Min_Price=("Modal Price", "min"),
        Max_Price=("Modal Price", "max"),
        Price_Observations=("Modal Price", "size"),
        Markets=("Market ID", "nunique"),
        States=("State ID", "nunique")
    )
)

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

monthly = monthly.sort_values(
    ["Crop", "Month Start"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

monthly.to_csv(
    MONTHLY_PRICE_FILE,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"\nMonthly rows created: {len(monthly):,}")

print(f"\nSaved to:")
print(MONTHLY_PRICE_FILE)

print("\nMonthly dataset columns:")
for column in monthly.columns:
    print(f" - {column}")

print("\nMonthly coverage by crop:")

coverage = (
    monthly
    .groupby("Crop")
    .agg(
        months=("Month Start", "size"),
        first_month=("Month Start", "min"),
        last_month=("Month Start", "max"),
        total_observations=("Price_Observations", "sum")
    )
    .sort_values("total_observations", ascending=False)
)

print(coverage.to_string())

print("\nFirst 20 monthly records:")
print(
    monthly.head(20).to_string(index=False)
)

print("\nMonthly crop price dataset created successfully.")

In [ ]:
import pandas as pd
from pathlib import Path

MONTHLY_PRICE_FILE = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "monthly_crop_prices.csv"
)

monthly = pd.read_csv(MONTHLY_PRICE_FILE)

monthly["Month Start"] = pd.to_datetime(
    monthly["Month Start"]
)

print("=" * 90)
print("MONTHLY COVERAGE GAP ANALYSIS")
print("=" * 90)

# ------------------------------------------------------------
# Intended 12-crop project set
# ------------------------------------------------------------

target_crops = [
    "Wheat",
    "Rice",
    "Maize",
    "Jowar/Sorghum",
    "Groundnut",
    "Cotton",
    "Banana",
    "Onion",
    "Potato",
    "Lentil",
    "Sweet Potato",
    "Bajra/Pearl Millet",
]

# ------------------------------------------------------------
# Expected monthly calendar
# ------------------------------------------------------------

expected_months = pd.date_range(
    start="2015-01-01",
    end="2025-12-01",
    freq="MS"
)

print(f"\nExpected months per crop: {len(expected_months)}")
print(f"Expected total crop-months: {len(target_crops) * len(expected_months):,}")

# ------------------------------------------------------------
# Coverage summary
# ------------------------------------------------------------

print("\nCoverage summary:")

coverage_rows = []

for crop in target_crops:

    actual = set(
        monthly.loc[
            monthly["Crop"] == crop,
            "Month Start"
        ]
    )

    expected = set(expected_months)

    missing = sorted(expected - actual)

    coverage_rows.append(
        {
            "Crop": crop,
            "Available Months": len(actual),
            "Missing Months": len(missing),
            "Coverage %": round(
                len(actual) / len(expected) * 100,
                2
            ),
            "First Available": min(actual) if actual else None,
            "Last Available": max(actual) if actual else None,
        }
    )

coverage = pd.DataFrame(coverage_rows)

print(
    coverage.to_string(index=False)
)

# ------------------------------------------------------------
# Detailed missing month list
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("MISSING CROP-MONTHS")
print("-" * 90)

for crop in target_crops:

    actual = set(
        monthly.loc[
            monthly["Crop"] == crop,
            "Month Start"
        ]
    )

    missing = sorted(
        set(expected_months) - actual
    )

    if missing:

        print(f"\n{crop}: {len(missing)} missing")

        # Group missing months by year
        missing_by_year = {}

        for date in missing:
            year = date.year
            missing_by_year.setdefault(year, []).append(date.month)

        for year, months in missing_by_year.items():
            month_names = ", ".join(
                pd.Timestamp(
                    year=year,
                    month=month,
                    day=1
                ).strftime("%b")
                for month in months
            )

            print(f"  {year}: {month_names}")

    else:
        print(f"\n{crop}: COMPLETE")

# ------------------------------------------------------------
# Current dataset crop list
# ------------------------------------------------------------

print("\n" + "-" * 90)
print("CROPS CURRENTLY PRESENT IN MONTHLY DATA")
print("-" * 90)

for crop in sorted(monthly["Crop"].unique()):
    print(f" - {crop}")

print("\nCoverage gap analysis complete.")

In [ ]:
import requests
import time

print("=" * 90)
print("TARGETED AGMARKNET AVAILABILITY CHECK")
print("=" * 90)

AGMARKNET_URL = "https://api.agmarknet.gov.in/v1/prices-and-arrivals/market-report/specific"

# Verified commodity mappings from our earlier investigation
test_crops = {
    "Potato": (24, 6),
    "Lentil": (52, 2),
    "Sweet Potato": (124, 6),
    "Bajra/Pearl Millet": (28, 1),
}

# One representative date per year.
test_dates = [
    "2015-01-01",
    "2017-01-01",
    "2019-01-01",
    "2021-01-01",
    "2023-01-01",
    "2025-01-01",
]

results = []

for crop, (commodity_id, group_id) in test_crops.items():

    print(f"\nTesting: {crop}")

    for date in test_dates:

        params = {
            "date": date,
            "commodityGroupId": group_id,
            "commodityId": commodity_id,
        }

        try:
            response = requests.get(
                AGMARKNET_URL,
                params=params,
                headers=agmarknet_headers,
                timeout=20
            )

            if response.status_code != 200:
                print(
                    f"  {date}: HTTP {response.status_code}"
                )

                results.append({
                    "Crop": crop,
                    "Date": date,
                    "Status": response.status_code,
                    "States": 0,
                    "Observations": 0
                })

                time.sleep(2)
                continue

            payload = response.json()

            states = payload.get("data", {}).get("states", [])

            observation_count = sum(
                len(state.get("data", []))
                for state in states
            )

            print(
                f"  {date}: "
                f"{len(states)} states, "
                f"{observation_count:,} observations"
            )

            results.append({
                "Crop": crop,
                "Date": date,
                "Status": response.status_code,
                "States": len(states),
                "Observations": observation_count
            })

        except Exception as e:

            print(
                f"  {date}: ERROR - {type(e).__name__}"
            )

            results.append({
                "Crop": crop,
                "Date": date,
                "Status": "ERROR",
                "States": 0,
                "Observations": 0
            })

        time.sleep(2)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

availability_test = pd.DataFrame(results)

print("\n" + "=" * 90)
print("AVAILABILITY TEST SUMMARY")
print("=" * 90)

print(
    availability_test.to_string(index=False)
)

print("\nSuccessful test dates by crop:")

for crop in test_crops:

    crop_results = availability_test[
        availability_test["Crop"] == crop
    ]

    successful = crop_results[
        (crop_results["Status"] == 200)
        & (crop_results["Observations"] > 0)
    ]

    print(
        f" - {crop}: "
        f"{len(successful)}/{len(test_dates)}"
    )

print("\nTargeted availability test complete.")

In [ ]:
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed/monthly_crop_prices.csv")

print("=" * 90)
print("PROCESSED FILE HEADER CHECK")
print("=" * 90)

print(f"\nFile exists: {processed_path.exists()}")
print(f"File path: {processed_path.resolve()}")

# Read only the header
header = pd.read_csv(processed_path, nrows=0)

print("\nColumns found:")
for i, column in enumerate(header.columns, start=1):
    print(f"{i}. {repr(column)}")

print("\nRaw first 3 lines of the file:")
with open(processed_path, "r", encoding="utf-8-sig") as f:
    for _ in range(3):
        line = f.readline()
        if not line:
            break
        print(repr(line[:500]))

In [ ]:
from pathlib import Path
import pandas as pd

processed_path = Path("../data/processed/monthly_crop_prices.csv")

df_monthly = pd.read_csv(
    processed_path,
    parse_dates=["Month Start"]
)

print("=" * 90)
print("CURRENT MONTHLY DATASET CHECK")
print("=" * 90)

print(f"\nRows: {len(df_monthly):,}")
print(f"Columns: {len(df_monthly.columns)}")

print("\nCrops:")
print(sorted(df_monthly["Crop"].unique()))

print("\nCoverage by crop:")

coverage = (
    df_monthly.groupby("Crop")
    .agg(
        Start_Date=("Month Start", "min"),
        End_Date=("Month Start", "max"),
        Months=("Month Start", "nunique"),
        Observations=("Price_Observations", "sum"),
        Markets=("Markets", "sum"),
        States=("States", "max")
    )
    .sort_index()
)

print(coverage.to_string())

print("\nMissing months within each crop's own date range:")

for crop, group in df_monthly.groupby("Crop"):
    expected_dates = pd.date_range(
        group["Month Start"].min(),
        group["Month Start"].max(),
        freq="MS"
    )

    actual_dates = pd.DatetimeIndex(group["Month Start"])

    missing = expected_dates.difference(actual_dates)

    print(f" - {crop}: {len(missing)} missing months")

print("\nFirst 10 rows:")
display(df_monthly.head(10))

In [ ]:
print("=" * 90)
print("FINAL PROCESSED DATA QUALITY CHECK")
print("=" * 90)

# ---------------------------------------------------------
# 1. Missing values
# ---------------------------------------------------------
print("\n1. MISSING VALUES")
missing = df_monthly.isna().sum()

if missing.sum() == 0:
    print("✓ No missing values found.")
else:
    print(missing[missing > 0].to_string())


# ---------------------------------------------------------
# 2. Duplicate Crop-Month records
# ---------------------------------------------------------
print("\n2. DUPLICATE CROP-MONTH RECORDS")

duplicates = df_monthly.duplicated(
    subset=["Crop", "Year", "Month"],
    keep=False
)

duplicate_count = duplicates.sum()

if duplicate_count == 0:
    print("✓ No duplicate Crop-Year-Month records found.")
else:
    print(f"⚠ {duplicate_count:,} duplicate rows found.")
    display(
        df_monthly.loc[duplicates]
        .sort_values(["Crop", "Year", "Month"])
        .head(20)
    )


# ---------------------------------------------------------
# 3. Non-positive prices
# ---------------------------------------------------------
print("\n3. PRICE VALIDATION")

price_columns = [
    "Median_Price",
    "Mean_Price",
    "Min_Price",
    "Max_Price"
]

for column in price_columns:
    count = (df_monthly[column] <= 0).sum()
    print(f" - {column}: {count:,} non-positive values")


# ---------------------------------------------------------
# 4. Logical price consistency
# ---------------------------------------------------------
print("\n4. PRICE CONSISTENCY CHECK")

invalid_min_max = (
    df_monthly["Min_Price"] > df_monthly["Max_Price"]
).sum()

invalid_median = (
    (df_monthly["Median_Price"] < df_monthly["Min_Price"]) |
    (df_monthly["Median_Price"] > df_monthly["Max_Price"])
).sum()

invalid_mean = (
    (df_monthly["Mean_Price"] < df_monthly["Min_Price"]) |
    (df_monthly["Mean_Price"] > df_monthly["Max_Price"])
).sum()

print(f" - Min > Max: {invalid_min_max:,}")
print(f" - Median outside Min-Max: {invalid_median:,}")
print(f" - Mean outside Min-Max: {invalid_mean:,}")


# ---------------------------------------------------------
# 5. Observation counts
# ---------------------------------------------------------
print("\n5. OBSERVATION COUNTS")

zero_observations = (
    df_monthly["Price_Observations"] <= 0
).sum()

print(f" - Months with zero price observations: {zero_observations:,}")

print("\nObservation statistics:")
print(
    df_monthly["Price_Observations"]
    .describe()
    .round(2)
    .to_string()
)


# ---------------------------------------------------------
# 6. Overall price statistics by crop
# ---------------------------------------------------------
print("\n6. PRICE STATISTICS BY CROP")

crop_stats = (
    df_monthly.groupby("Crop")["Median_Price"]
    .agg(
        Mean="mean",
        Median="median",
        Minimum="min",
        Maximum="max",
        Std_Dev="std"
    )
    .round(2)
)

print(crop_stats.to_string())


print("\n" + "=" * 90)
print("DATA QUALITY CHECK COMPLETE")
print("=" * 90)